# NumPy를 이용한 선형 회귀 모델 구현 - 실습

---

## 학습 개요

1. **학습 주제**

   - 데이터 전처리 및 정규화
   - 선형 회귀 모델 학습
   - 경사 하강법 기반 최적화

2. **학습 목표**

   - 표준 정규화 과정을 구현하여 입력 데이터의 스케일을 조정할 수 있다.
   - `np.linalg.lstsq`와 SVD를 활용해 선형 회귀의 최적 파라미터를 구할 수 있다.
   - 경사 하강법으로 모델을 학습하고 MSE 손실을 계산할 수 있다.
   - NumPy의 브로드캐스팅과 행렬 연산(`@`, `.T`, `shape`)을 이해하고 활용할 수 있다.
   - 학습 과정에서 손실 함수의 변화를 시각화하고 해석할 수 있다.

3. **핵심 개념**

   - **표준화(Standard Scaling)**: 평균 0, 분산 1로 각 특성을 변환하여 학습 안정성을 높이는 기법
   - **브로드캐스팅(Broadcasting)**: 서로 다른 shape의 배열 간 연산 시 자동으로 차원을 확장하여 계산하는 NumPy 기능
   - **최소제곱법(Least Squares)**: 예측값과 실제값의 차이(잔차)의 제곱합을 최소화하는 최적화 원리
   - **특이값 분해(SVD)**: 행렬을 U, Σ, Vᵀ로 분해하여 안정적인 유사역행렬을 구하는 기법
   - **경사 하강법(Gradient Descent)**: 손실 함수의 기울기를 따라 파라미터를 반복적으로 업데이트하여 최적값을 찾는 알고리즘
   - **손실 함수(MSE)**: 모델의 예측이 얼마나 틀렸는지를 수치화한 목표값으로, 이를 줄이는 방향으로 학습이 진행됨

4. **선행 지식**

   - Python 기본 문법 및 NumPy 기초
   - 평균, 표준편차 등 기초 통계 개념
   - (선택) 벡터와 행렬의 기본 개념 (없어도 실습 가능하도록 설명 제공)

## 실습 구성

1. **학습 방향**

   - **실습 구성 방식**
     - 각 단계별로 TODO 영역을 채우며 학습자가 직접 구현합니다.

   - **Required Package**
     ```
     python>=3.11
     numpy>=2.0.0
     pandas>=2.0.0
     matplotlib>=3.8.0
     seaborn>=0.13.2
     tqdm>=4.66.0
     ```

   - **Step 요약**
     - **Step 1**: 데이터 로딩 및 정규화 - 연속형 변수 추출, 평균·표준편차 계산을 통한 표준 정규화 수행
     - **Step 2**: 선형대수 기반 해법 - 정규방정식, `np.linalg.lstsq`, SVD를 활용한 최적 파라미터 계산
     - **Step 3**: 경사 하강법 학습 - 손실 함수(MSE) 계산, 그래디언트 업데이트, 학습 곡선 시각화
     - **Step 4**: 미니배치 학습 및 조기 종료 - 배치 단위 학습, Early Stopping, Gradient Accumulation 구현

2. **데이터셋 개요 및 저작권 정보**

   - **데이터셋 명**: Diamonds 데이터셋
   - **데이터셋 개요**: 약 54,000개의 다이아몬드 가격(`price`)과 주요 속성(`carat`, `depth`, `table`, `x`, `y`, `z`)을 포함한 데이터
   - **사용 목적**: 연속형 변수를 이용한 선형 회귀 모델 학습 및 예측 성능 평가
   - **저작권/출처**: seaborn-data 저장소(BSD 라이선스)에 따라 제공
   - **주의사항**: 실습 목적의 데이터로, 실제 다이아몬드 가격 예측에는 추가적인 특성 공학과 모델 검증이 필요

3. **문제 설명**

   - **문제 개요**: 이 실습은 **NumPy를 활용한 선형 회귀 모델의 직접 구현**을 통해 머신러닝의 기초 원리를 익히기 위해 설계되었습니다. 학습자는 **데이터 정규화, 선형대수 기반 최적해 계산, 경사 하강법**을 코드로 확인하고, 최종적으로 **모델을 학습하고 평가하며 학습 과정을 시각화**할 수 있어야 합니다.

   - **요구사항 요약**
     - **데이터 전처리**: 연속형 변수 추출 및 표준 정규화 수행
     - **해석적 해법**: 정규방정식, lstsq, SVD를 사용한 파라미터 계산
     - **반복적 최적화**: 경사 하강법을 구현하여 손실을 최소화
     - **학습 안정성**: 미니배치 학습, 조기 종료, Gradient Accumulation 적용
     - **시각화**: matplotlib을 활용한 예측 결과 및 학습 곡선 시각화

4. **학습 문제: Step–TODO 구체 설명**
   - **Step 1 — 데이터 정규화 및 준비**
     - **TODO 1:** `diamonds` 데이터셋에서 **연속형 변수와 범주형 변수를 각각 분리**하여 리스트에 저장하기 (*연결 핵심개념: 데이터 전처리, 브로드캐스팅 활용*)
     - **TODO 2:** 연속형 변수의 평균과 표준편차를 구해 **표준화(Standardization) 전처리** 수행하기(*연결 핵심개념: 표준화(Standard Scaling), 브로드캐스팅*)
     - **1줄 요약**: 입력 데이터의 스케일 차이를 줄여 학습 안정성을 높인다

   - **Step 2 — 선형대수를 이용한 해법**
     - **TODO 3:** 행렬 연산을 통해 **정규방정식(Normal Equation)을 직접 구현**하고 최적 파라미터 구하기 (*연결 핵심개념: 최적 파라미터 계산, 수치적 안정성 이해*)
     - **TODO 4:** NumPy의 **`np.linalg.lstsq` 함수를 활용**하여 선형 회귀 파라미터 구하기 (*연결 핵심개념: 최소제곱법, 특이값 분해(SVD)*)
     - **TODO 5:** **특이값 분해(SVD)**를 통해 유사역행렬을 계산하고 이를 이용해 파라미터 구하기 (*연결 핵심개념: 최소제곱법, 특이값 분해(SVD)*)
     - **1줄 요약**: 행렬 연산을 통해 한 번에 최적 파라미터를 구하는 방법을 익힌다

   - **Step 3 — 경사 하강법과 손실 계산**
     - **TODO 6:** 반복문을 통해 파라미터를 업데이트하는 **경사 하강법(Gradient Descent) 알고리즘 구현**하기(*연결 핵심개념: 경사 하강법 구현, 손실 함수 계산*)
     - **1줄 요약**: 반복적으로 파라미터를 업데이트하며 손실을 줄여가는 학습 과정을 구현한다

   - **Step 4 — 미니배치 학습 및 조기 종료**
     - **TODO 7:** **미니배치(Mini-batch) 처리와 조기 종료(Early Stopping)** 기능이 포함된 학습 함수 구현하기(*연결 핵심개념: 효율적인 학습 기법 적용*)
     - **TODO 8:** **다양한 학습률(Learning Rate)** 값을 적용해보며 학습 과정과 결과의 변화 실험하기(*연결 핵심개념: Mini-batch SGD, Early Stopping, Gradient Accumulation*)
     - **1줄 요약**: 대규모 데이터에서도 효율적으로 학습할 수 있는 기법들을 적용한다

## Concept Check: ML 기초 지식 - 학습이란 무엇인가?

머신러닝에서 **학습(Training)**이란 모델이 데이터로부터 패턴을 찾아내는 과정입니다. 이 과정은 크게 **4단계**로 나뉩니다:

1. **입력 → 예측 → 손실 → 업데이트**
   1. **입력(Input)**: 데이터를 모델에 넣습니다 (예: 다이아몬드의 크기, 깊이 등)
   2. **예측(Prediction)**: 모델이 현재 파라미터로 값을 예측합니다 (예: 가격 예측)
   3. **손실(Loss)**: 예측값과 실제값이 얼마나 다른지 계산합니다
   4. **업데이트(Update)**: 손실을 줄이는 방향으로 파라미터를 조정합니다

   이 4단계를 반복하면서 모델은 점점 더 정확한 예측을 하게 됩니다.


2. 손실 함수(Loss Function)란?   
   **손실 함수**는 "모델이 얼마나 틀렸는가"를 수치화한 목표값입니다.
   - 예측이 정확하면 → 손실이 **작아집니다** (0에 가까워짐)
   - 예측이 틀리면 → 손실이 **커집니다**

   **왜 손실을 줄여야 할까요?**  
   손실을 줄인다 = 예측을 더 정확하게 만든다는 의미이기 때문입니다. 학습의 목표는 이 손실 함수를 최소화하는 것입니다.   
   선형 회귀에서는 주로 **MSE(Mean Squared Error, 평균 제곱 오차)**를 손실 함수로 사용합니다:

   $$
   \text{MSE} = \frac{1}{m}\sum_{i=1}^m (\text{예측값}_i - \text{실제값}_i)^2
   $$

   - 차이를 제곱하는 이유: 크게 틀린 예측에 더 큰 패널티를 주기 위함
   - 평균을 내는 이유: 데이터 개수에 상관없이 비교하기 위함


3. 활성화 함수(Activation Function)란?
   **활성화 함수**는 모델의 출력을 비선형으로 변환하는 함수입니다. 예를 들어:   
   - **시그모이드(Sigmoid)**: 출력을 0~1 사이로 변환 (확률 예측에 사용)
   - **ReLU**: 음수를 0으로 만들고 양수는 그대로 (딥러닝에서 많이 사용)

   **선형 회귀에는 활성화 함수가 없습니다!**.  
   - 선형 회귀는 입력의 가중합을 그대로 출력으로 사용합니다
   - 활성화 함수는 주로 **분류 문제**(로지스틱 회귀, 신경망 등)에서 사용됩니다
   - 예: 로지스틱 회귀는 시그모이드 활성화 함수를 사용해 0~1 사이의 확률을 출력합니다

   즉, **활성화 함수 = 항상 쓰는 것이 아니라, 문제 유형에 따라 선택적으로 사용**하는 것입니다.


4. NumPy 행렬 연산 기초

   선형대수 지식이 없어도 괜찮습니다! 다음 세 가지만 기억하세요:

   1. **`@` (행렬 곱셈)**: 두 배열을 곱할 때 사용
      - 예: `y_pred = X @ theta` → 입력 데이터와 파라미터를 곱해 예측값 계산
      - 중요: 앞 배열의 열 개수 = 뒤 배열의 행 개수여야 함

   2. **`.T` (전치)**: 행과 열을 바꿈
      - 예: `X.T @ error` → 열 방향으로 계산하던 것을 행 방향으로 바꿔서 계산
      - shape (100, 5) → (5, 100)으로 변환

   3. **`.shape` (크기 확인)**: 배열의 차원 확인
      - 예: `X.shape = (100, 5)` → 100개 데이터, 5개 특성
      - 디버깅 시 shape 확인은 필수!

   **직관적 이해**:
   - `X @ theta`: "각 데이터에 대해 (특성 × 가중치)를 모두 더한 값"
   - `X.T @ error`: "각 특성이 오차에 얼마나 기여했는지 계산"

   shape만 맞추면 코드가 작동합니다!

## Install & Import

In [ ]:
# 공통 실습 환경 설치 (최초 1회 실행)
!pip install -q \
    "numpy>=2.0.0" \
    "pandas>=2.0.0" \
    "matplotlib>=3.8.0" \
    "seaborn>=0.13.2" \
    "tqdm>=4.66.0"


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import List
import pandas as pd
sns.set_theme()

## Step 1: 데이터 정규화 및 준비

### Concept Check

#### 왜 데이터 정규화가 필요할까요?

모든 데이터 분석의 첫 번째는 무엇일까요? 당연히 데이터 수집이 1순위이지만, 수집이 끝난 뒤 가장 먼저 할 일은 **데이터 전처리**입니다. 오늘은 seaborn 라이브러리의 `diamonds` 데이터셋을 사용해, 실제 수치형 특성만 뽑아 선형 회귀 모델을 준비하고 학습해 보는 과정을 살펴봅니다.

#### Feature의 종류

우리는 데이터 한 건마다 갖게 되는 속성들을 **feature**(특성)라고 부릅니다. 일반적으로 한 건의 데이터는 행(row)으로 표현되고, 하나의 feature는 열(column)에 해당합니다.  

feature는 성격에 따라 크게 **연속형 변수**와 **범주형 변수**로 나눌 수 있습니다:

- **연속형 변수(Continuous Variable)**  
  - 수치가 연속적인 스펙트럼을 이루며, 실수 형태로 측정됩니다
  - 예시: 키(cm), 몸무게(kg), 가격(USD), 온도(°C), 시간 간격(초)  
  - 전처리: 표준화(Standardization), 정규화(Normalization) 등으로 스케일 조정

- **범주형 변수(Categorical Variable)**
  - 유한 개의 범주 또는 등급으로 구분됩니다
  - 명목형(Nominal): 성별, 혈액형, 색상 (순서 없음)
  - 순서형(Ordinal): 만족도, 학위 (순서 있음)
  - 전처리: 원-핫 인코딩, 순서형 인코딩, 임베딩 등으로 숫자화

#### 표준화(Standardization)란?

표준화는 데이터의 분포를 **평균 0, 표준편차 1**이 되도록 변환하는 전처리 기법입니다.

$$
Z = \frac{x - \mu}{\sigma}
$$

- $\mu$: 평균 (mean)
- $\sigma$: 표준편차 (standard deviation)

**왜 표준화가 필요할까요?**

1. **스케일 차이 해소**: 서로 다른 단위(예: 크기는 0\~10, 가격은 0\~20000)를 가진 특성들을 동등하게 만듦
2. **학습 속도 향상**: 경사 하강법이 더 빠르고 안정적으로 수렴
3. **수치 안정성**: 매우 큰 값이나 작은 값으로 인한 계산 오류 방지
4. **거리 기반 알고리즘**: KNN, K-means 등에서 특정 특성이 지배하는 것을 방지

### Guided Build: 데이터 살펴보기

In [ ]:
# 데이터셋 로딩
df = sns.load_dataset("diamonds")

In [ ]:
# 데이터셋의 구조 확인
df.info()

In [ ]:
# 기술 통계량 확인
df.describe()

### TODO 1: 연속형/범주형 변수 분류하기

`diamonds` dataset에서 연속형 변수와 범주형 변수를 분리해봅시다.

- **요구사항**:
  - 범주형 변수(dtype='category')를 `categorical_cols` 리스트에 저장
  - 연속형 변수(숫자형 dtype)를 `continuous_cols` 리스트에 저장
- **힌트**: `df.select_dtypes()` 메서드 활용 ([문서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html))

In [ ]:
# 범주형 변수 추출 (dtype이 'category'인 열)
categorical_cols: List[str] = None
# 연속형 변수 추출 (숫자형 dtype인 열)
continuous_cols: List[str] = None

# 정답 확인용 코드 (수정하지 마세요)
expected_cats = {"cut", "color", "clarity"}
expected_nums = {"carat", "depth", "table", "price", "x", "y", "z"}

assert len(categorical_cols) == 3, f"범주형 변수 개수는 3이어야 합니다 (현재 {len(categorical_cols)})"
assert len(continuous_cols) == 7, f"연속형 변수 개수는 7이어야 합니다 (현재 {len(continuous_cols)})"
assert set(categorical_cols) == expected_cats, f"범주형 컬럼이 예상과 다릅니다: {categorical_cols}"
assert set(continuous_cols) == expected_nums, f"연속형 컬럼이 예상과 다릅니다: {continuous_cols}"

print("✅ 모든 테스트를 통과했습니다!")
print(f"범주형 변수: {categorical_cols}")
print(f"연속형 변수: {continuous_cols}")

### TODO 2: 표준화 진행하기

연속형 변수들을 표준화(평균 0, 표준편차 1)해봅시다.

- **요구사항**:
  1. 연속형 변수만 포함하는 `X_raw` 배열 생성
  2. 각 열의 평균 `mu` 계산
  3. 각 열의 표준편차 `std` 계산 (0인 경우 1로 대체하여 안전장치)
  4. 브로드캐스팅을 활용해 `(X_raw - mu) / std` 형태로 표준화하여 `X_norm` 생성
- **힌트**: `np.mean()`, `np.std()`, `np.where()` 활용

In [ ]:
# 1. 연속형 변수만 담긴 X_raw 생성
X_raw: np.ndarray = None

# 2. 각 열의 평균 계산
mu: np.ndarray = None

# 3. 각 열의 표준편차 계산 (0인 경우 1로 대체)
std: np.ndarray = None

# 4. 브로드캐스팅을 활용한 표준화
X_norm: np.ndarray = None

# 정답 확인용 코드 (수정하지 마세요)
# Shape 확인
assert X_norm.shape == X_raw.shape, f"Shape mismatch: {X_norm.shape}"

# 결과의 통계적 특성 확인 (평균=0, 표준편차=1 인지 검사)
# atol(절대 오차 허용 범위)을 사용하여 부동소수점 오차 허용
output_mean = np.mean(X_norm, axis=0)
output_std = np.std(X_norm, axis=0)

# 모든 열의 평균이 0에 매우 가까운지 확인
np.testing.assert_allclose(output_mean, 0, atol=1e-6, err_msg="표준화된 데이터의 평균이 0이 아닙니다.")

# 모든 열의 표준편차가 1에 매우 가까운지 확인 (단, 원본 std가 0이었던 컬럼은 결과도 0일 수 있음을 고려해야 함)
# 원본 데이터에 상수 컬럼(모든 값이 동일)이 없다면 아래 코드로 충분합니다.
np.testing.assert_allclose(output_std, 1, atol=1e-6, err_msg="표준화된 데이터의 표준편차가 1이 아닙니다.")

print("✅ 정답입니다! (평균 0, 표준편차 1 검증 완료)")

## Step 2: 선형대수를 이용한 해법

### Concept Check

#### 선형회귀란?

**선형 회귀(Linear Regression)**는 입력 특성들의 가중합(weighted sum)으로 출력을 예측하는 모델입니다.

$$
\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n
$$

- $\hat{y}$: 예측값 (predicted value)
- $\theta_0$: 절편 (intercept) - 모든 입력이 0일 때의 기본값
- $\theta_1, \theta_2, \ldots, \theta_n$: 가중치 (weights) - 각 특성의 중요도
- $x_1, x_2, \ldots, x_n$: 입력 특성들

**행렬 형태로 표현하면:**

$$
\hat{y} = X_b \cdot \theta
$$

- $X_b$: 첫 열이 모두 1인 설계 행렬 (design matrix) - shape: (m, n+1)
- $\theta$: 파라미터 벡터 - shape: (n+1,)
- m: 샘플 수, n: 특성 수

#### 정규방정식(Normal Equation)

선형 회귀의 최적 파라미터는 수학적으로 다음과 같이 구할 수 있습니다:

$$
\theta = (X_b^T X_b)^{-1} X_b^T y
$$

**장점**: 한 번의 계산으로 최적해를 구할 수 있음  
**단점**:
- 역행렬 계산이 불안정할 수 있음
- 특성 수가 많으면 계산량이 많음 ($O(n^3)$)

#### 최소제곱법(Least Squares): `np.linalg.lstsq`

정규방정식의 단점을 보완한 방법입니다:
- SVD(특이값 분해)를 내부적으로 사용하여 수치적으로 안정적
- 역행렬이 존재하지 않아도 해를 구할 수 있음
- NumPy가 제공하는 최적화된 구현

#### SVD(특이값 분해)

행렬을 세 개의 행렬로 분해하는 기법:

$$
X_b = U \Sigma V^T
$$

이를 이용하면 안정적인 유사역행렬(pseudo-inverse)을 구할 수 있습니다:

$$
\theta = V \Sigma^+ U^T y
$$

- $\Sigma^+$: $\Sigma$의 역수로 만든 대각 행렬

### TODO 3: 정규방정식으로 파라미터 구하기

정규방정식 $\theta = (X_b^T X_b)^{-1} X_b^T y$를 단계별로 구현해봅시다.

- **요구사항**:
  1. feature `X`와 target `y` 분리
  2. 절편항(모두 1인 열)을 추가하여 `X_b` 생성
  3. $X_b^T X_b$를 계산하여 `XT_X`에 저장
  4. $X_b^T y$를 계산하여 `XT_y`에 저장
  5. 역행렬과 행렬 곱으로 `theta` 계산
  6. 얻어낸 해 `theta`를 통해 예측값 `y_pred` 만들기
- **힌트**: `np.c_[]`, `@`, `np.linalg.inv()` 활용

In [ ]:
# 편의를 위해 X_norm을 DataFrame으로 변환
X_df = pd.DataFrame(X_norm, columns=continuous_cols)

# 1. X와 y 분리
X: np.ndarray = None
y: np.ndarray = None

# 2. 절편항 추가
m, n = X.shape
X_b = None  # shape = (m, n+1)

# 3. X_b^T @ X_b 계산
XT_X = None  # shape = (n+1, n+1)

# 4. X_b^T @ y 계산
XT_y = None  # shape = (n+1,)

# 5. 역행렬을 이용한 theta 계산
theta = None # shape = (n+1,)

# 6. `theta`를 통한 해 계산
y_pred = None

print("정규방정식으로 구한 파라미터 θ:")
print(theta)
print(f"\nX_b shape: {X_b.shape}")
print(f"theta shape: {theta.shape}")

### Test (Checkpoint)

구한 파라미터로 예측을 수행하고 시각화해봅시다.

In [ ]:
# 시각화 함수 (재사용)
def plot_prediction(y_true: np.ndarray, y_pred: np.ndarray, title: str = "Actual vs Predicted") -> None:
    """예측값과 실제값을 산점도로 시각화"""
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    assert y_true.shape == y_pred.shape, f"Size mismatch between y_true and y_pred"

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5, label="Model Prediction", ax=ax)
    sns.lineplot(x=[y.min(), y.max()], y=[y.min(), y.max()],
                 label="Ideal Regression", linestyle="--", color="red")
    ax.set_xlabel('Actual Price')
    ax.set_ylabel('Predicted Price')
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

# 예측값을 통한 MSE 계산
mse = np.mean((y_pred - y) ** 2)

print(f"정규방정식 MSE: {mse:.6f}")
plot_prediction(y_true=y, y_pred=y_pred, title=f"Normal Equation Predicton Result\nMSE: {mse:.4f}")

### TODO 4: lstsq로 파라미터 구하기

`np.linalg.lstsq`를 사용하여 더 안정적으로 파라미터를 구해봅시다.

- **요구사항**: `np.linalg.lstsq(X_b, y, rcond=None)`를 호출하여 `theta_lstsq` 구하기
- **힌트**: lstsq는 4개의 값을 반환 (해, 잔차, 랭크, 특이값)

In [ ]:
theta_lstsq, residuals, rank, singular_vals = None

# 예측 및 평가
y_pred = X_b @ theta_lstsq
mse = np.mean((y_pred - y) ** 2)

print(f"lstsq 파라미터 θ: {theta_lstsq}")
print(f"lstsq MSE: {mse:.6f}")
plot_prediction(y_true=y, y_pred=y_pred, title=f"lstsq Prediction Result\nMSE: {mse:.4f}")

### TODO 5: SVD로 파라미터 구하기

SVD를 직접 사용하여 파라미터를 구해봅시다.

- **요구사항**:
  1. `np.linalg.svd(X_b, full_matrices=False)`로 U, S, Vt 구하기
  2. S의 역수로 대각행렬 `S_plus` 만들기 (힌트: `np.diag(1.0 / S)`)
  3. $\theta = V^T \Sigma^+ U^T y$ 계산
- **힌트**: Vt는 이미 전치된 상태이므로 `Vt.T`를 사용

In [ ]:
# 1. SVD 수행
U, S, Vt = None

# 2. Pseudo-inverse S_plus 생성
S_plus = None

# 3. theta 계산: V @ S_plus @ U.T @ y
theta_svd = None

# 예측 및 평가
y_pred = X_b @ theta_svd
mse = np.mean((y_pred - y) ** 2)

print(f"SVD 파라미터 θ: {theta_svd}")
print(f"SVD MSE: {mse:.6f}")
plot_prediction(y_true=y, y_pred=y_pred, title=f"SVD Prediction Result\nMSE: {mse:.4f}")

### 📝 참고: 세 방법의 결과 비교

세 가지 방법(정규방정식, lstsq, SVD)으로 구한 파라미터가 거의 동일함을 확인할 수 있습니다. 이는 모두 같은 최적화 문제를 풀고 있기 때문입니다. 다만 수치적 안정성과 계산 효율성에서 차이가 있습니다.

## Step 3: 경사 하강법과 손실 계산

### Concept Check

#### 왜 경사 하강법을 사용해야 할까요?

앞에서 본 선형대수 기반 방법들은:
- 한 번에 최적해를 구할 수 있지만
- 데이터가 많거나 특성이 많으면 메모리와 계산량이 급격히 증가
- 역행렬 계산이 필요 ($O(n^3)$ 복잡도)

**경사 하강법(Gradient Descent)** 은:
- 반복적으로 조금씩 파라미터를 업데이트
- 대규모 데이터에서도 효율적
- 미니배치를 사용하면 메모리 사용량 감소
- 딥러닝 등 다양한 모델에 적용 가능

#### 경사 하강법의 원리

**1. 현재 위치에서 손실 함수의 기울기(그래디언트) 계산**

$$
\nabla_\theta = \frac{2}{m} X_b^T (X_b \theta - y)
$$

**2. 기울기의 반대 방향으로 파라미터 업데이트**

$$
\theta \leftarrow \theta - \alpha \nabla_\theta
$$

- $\alpha$: 학습률(learning rate) - 한 번에 얼마나 이동할지 결정
- 기울기가 양수면 → theta를 감소
- 기울기가 음수면 → theta를 증가
- 이렇게 하면 손실이 줄어드는 방향으로 이동!

#### 손실 함수(MSE) 계산

$$
\text{MSE} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)^2
$$

학습 과정에서 MSE를 추적하면:
- 학습이 잘 되고 있는지 확인
- 언제 멈춰야 할지 판단 (조기 종료)
- 학습률이 적절한지 확인

### TODO 6: 경사 하강법 구현하기

초기화된 파라미터를 기반으로 배치 경사 하강법(전체 데이터 사용)을 구현해봅시다.

- **요구사항**:
  1. iterations 횟수만큼 반복:
     - 현재 `theta`로 예측값 계산
     - 오차(error) 및 MSE 계산
     - 그래디언트 계산
     - `theta` 업데이트
     - `loss_history`에 MSE 저장
  2. 학습 곡선 시각화
- **힌트**: `tqdm`으로 진행 상황 표시

In [ ]:
# 초기 파라미터
theta = np.zeros(X_b.shape[1])  # shape = (n+1,)
alpha = 0.01  # 학습률
iterations = 1000  # 반복 횟수
loss_history = []  # 손실 기록

# 경사 하강법 루프
for i in tqdm(range(iterations), desc="Training"):
    # 1. 예측값 계산
    y_pred = None

    # 2. 오차 및 MSE 계산
    error = None
    mse = None
    loss_history.append(mse)

    # 3. 그래디언트 계산
    gradient = None

    # 4. theta 업데이트
    theta -= None

# 결과 출력
print(f"최종 θ: {theta}")
print(f"최종 MSE: {loss_history[-1]:.6f}")

# 학습 곡선 시각화
plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("Mean Squared Error", fontsize=12)
plt.title("Loss Curve (Gradient Descent)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

### Test (Checkpoint)

경사 하강법으로 학습한 모델의 예측 결과를 확인해봅시다.

In [ ]:
# 예측 및 시각화
y_pred_gd = X_b @ theta
mse_gd = np.mean((y_pred_gd - y) ** 2)

print(f"경사 하강법 최종 MSE: {mse_gd:.6f}")
plot_prediction(y_true=y, y_pred=y_pred_gd, title=f"Gradient Descent Prediction Result\nMSE: {mse_gd:.4f}")

### 📝 참고: 학습 과정에 대한 질문들

#### ❓ 초기설정값, 저렇게 하는게 맞나요?

- Iterations, 학습률(alpha), 초기 파라미터값 등 모두 "정답"은 없습니다
- 경험을 통해 좋은 값을 찾는 과정이 필요합니다
- 특히 **학습률**은 학습에 가장 큰 영향을 줍니다
  - 너무 크면: 발산(diverge)
  - 너무 작으면: 학습이 너무 느림
- 이런 학습되지 않는 값들을 **초매개변수(Hyperparameters)** 라고 합니다

#### ❓ iterations 전에 목표 MSE에 도달하면?

- 맞습니다! 조기 종료(Early Stopping)를 사용할 수 있습니다
- 다음 Step에서 구현해봅시다

#### ❓ 미분을 통한 업데이트, 최선인가?

- 배치 경사 하강법은 기본적인 방법입니다
- 한계:
  - 1차 도함수만 사용 → 학습률 설정이 까다로움
  - 전체 데이터 사용 → 메모리 부담
- 개선 방법들:
  - **모멘텀(Momentum)**: 이전 방향을 고려하여 관성 적용
  - **RMSProp**: 학습률을 자동으로 조절
  - **Adam**: 모멘텀 + RMSProp의 장점 결합
  - 이런 고급 옵티마이저들은 과제에서 다룹니다!

#### ❓ 학습할 때마다 이 코드를 매번 새로 작성하나요?

- 실무에서는 라이브러리를 사용합니다:
  - `scikit-learn`: 머신러닝 기본 모델
  - `PyTorch`, `TensorFlow`: 딥러닝 프레임워크
- 이들은 학습 루프, 손실 추적, 조기 종료, 다양한 옵티마이저를 제공
- 하지만 원리를 이해하기 위해 처음엔 직접 구현하는 것이 중요!

#### ❓ 데이터가 많을 때도 가능할까?

- 배치 경사 하강법은 전체 데이터를 한 번에 사용 → 메모리 부담
- 해결책:
  - **미니배치 경사 하강법**: 일부 데이터만 사용하여 업데이트
  - **확률적 경사 하강법(SGD)**: 한 번에 1개씩 사용
  - GPU 가속, 분산 처리
- 다음 Step에서 미니배치를 구현합니다!

#### ❓ 새로운 데이터에 대해서도 잘될까?

- 현재는 모든 데이터로 학습했습니다
- 문제: 학습 데이터에만 과적합(overfitting)될 수 있음
- 해결책:
  - **데이터 분할**: Train / Validation / Test
  - Train으로 학습, Validation으로 검증, Test로 최종 평가
- 이 내용은 Chapter 1에서 본격적으로 다룹니다

## Step 4: 미니배치 학습 및 조기 종료

### Concept Check

#### Mini-batch Gradient Descent

전체 데이터가 아닌 일부(batch)만 사용하여 그래디언트를 계산하고 업데이트합니다.

**장점**:
- 메모리 사용량 감소
- 학습 속도 향상 (더 자주 업데이트)
- 지역 최솟값(local minimum)에서 탈출 가능 (노이즈 효과)

**배치 크기 선택**:
- 너무 작으면: 노이즈가 커져 불안정
- 너무 크면: 메모리 부담, 지역 최솟값에 갇힐 위험
- 일반적으로 32, 64, 128, 256 등 사용

#### Gradient Accumulation

여러 미니배치의 그래디언트를 모아서 한 번에 업데이트하는 기법입니다.

**사용 이유**:
- 메모리는 절약하면서 큰 배치 효과를 얻음
- `effective_batch_size = batch_size × accumulate_steps`
- 분산 학습에서 통신 오버헤드 감소

**작동 방식**:
1. N개의 미니배치에서 그래디언트 계산
2. 그래디언트를 누적(accumulate)
3. N개가 모이면 평균 내어 한 번 업데이트

#### Early Stopping

손실이 충분히 작아지면 학습을 조기에 종료하는 기법입니다.

**장점**:
- 불필요한 계산 방지
- 과적합 방지 (validation loss 기준 시)
- 학습 시간 단축

**사용 방법**:
- 목표 손실값(`tol`) 설정
- 매 epoch마다 손실 확인
- `loss < tol`이면 중단

### TODO 7: 미니배치 학습 함수 구현하기

조기 종료, 미니배치, Gradient Accumulation을 모두 지원하는 학습 함수를 만들어봅시다.

- **요구사항**:
  1. 순차적으로 미니배치 구성 (start:end 인덱싱)
  2. 각 배치마다 그래디언트 계산 및 누적
  3. `accumulate_steps`만큼 모이면 파라미터 업데이트
  4. epoch마다 전체 데이터 MSE 계산
  5. `tol` 기준으로 조기 종료
- **힌트**:
  - `range(0, m, batch_size)`로 배치 시작점 생성
  - `min(start + batch_size, m)`으로 마지막 배치 처리

In [ ]:
def train_linear_regression_sequential(
    X_b, y,
    alpha=0.01,
    epochs=100,
    tol=None,
    batch_size=None,
    accumulate_steps=1,
):
    """
    순차적 미니배치 선형 회귀 학습 함수

    Parameters:
    -----------
    X_b : ndarray, shape (m, n+1)
        절편항이 포함된 입력 행렬
    y : ndarray, shape (m,)
        타깃 벡터
    alpha : float
        학습률
    epochs : int
        전체 반복 횟수
    tol : float or None
        조기 종료 임계값 (MSE가 tol 이하면 중단)
    batch_size : int or None
        미니배치 크기 (None이면 전체 배치)
    accumulate_steps : int
        Gradient Accumulation할 배치 수

    Returns:
    --------
    theta : ndarray
        학습된 파라미터
    history : list
        각 epoch의 MSE 기록
    """
    m, n = X_b.shape
    theta = np.zeros(n)  # 파라미터 초기화
    history = []

    # batch_size가 None이면 전체 데이터 사용
    bs = batch_size or m

    for epoch in range(1, epochs + 1):
        grad_accum = np.zeros_like(theta)  # 그래디언트 누적용
        accum_count = 0  # 누적된 배치 수

        # 순차적 미니배치 학습
        for start in range(0, m, bs):
            # 1. Mini-batch 구성
            end = min(start + bs, m)
            X_batch = None
            y_batch = None

            # 2. 배치에서 그래디언트 계산
            y_pred = None
            error = None
            grad = None

            # 3. 그래디언트 누적
            grad_accum = None
            accum_count = None

            # 4. accumulate_steps만큼 모이면 업데이트 및 `grad_accum`, `accum_count` 초기화
            if accum_count == accumulate_steps or end == m:
                theta = None
                grad_accum[:] = None
                accum_count = None

        # 5. 전체 데이터 MSE 계산 (epoch마다)
        mse = None
        history.append(mse)
        print(f"Epoch {epoch:3d}/{epochs}    MSE: {mse:.6f}")

        # 6. Early stopping (if 문을 완성해주세요)
        if tol is not None and mse < tol:
            print(f"-> Early stopping at epoch {epoch}, MSE={mse:.6f}")
            break

    return theta, history

In [ ]:
# 미니배치 학습 실행
theta_mb, loss_hist = train_linear_regression_sequential(
    X_b, y,
    alpha=0.01,
    epochs=100,
    tol=1e-4,
    batch_size=32,
    accumulate_steps=2
)

print(f"\n최종 추정된 θ: {theta_mb}")

# 학습 곡선 시각화
plt.figure(figsize=(10, 4))
plt.plot(loss_hist)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Mean Squared Error", fontsize=12)
plt.title("Loss Curve (Mini-batch with Early Stopping)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

# 최종 예측 시각화
y_pred_mb = X_b @ theta_mb
plot_prediction(y_true=y, y_pred=y_pred_mb, title="Mini-batch Prediction Result")

### TODO 8: 학습률 변경 실험

학습률을 변경하면 학습이 어떻게 달라질까요? 실험해봅시다.

- **요구사항**: `alpha` 값을 0.05, 1.0, 1e-5로 각각 변경하여 학습하고 결과 비교
- **관찰 포인트**:
  - 학습 속도 (수렴 속도)
  - 안정성 (발산 여부)
  - 최종 MSE

In [ ]:
learning_rates = [1e-5, 0.01, 0.05, 1.0]
results = {}

plt.figure(figsize=(12, 4))

for lr in learning_rates:
    print(f"\n{'='*50}")
    print(f"학습률 alpha = {lr}")
    print(f"{'='*50}")

    theta_exp, loss_exp = None
    results[lr] = (theta_exp, loss_exp)
    plt.plot(loss_exp, label=f"lr={lr}")

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("MSE", fontsize=12)
plt.title("Learning Curves of Different Learning Rates", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')  # 로그 스케일로 보면 차이가 더 명확
plt.show()

print("\n" + "="*50)
print("실험 요약")
print("="*50)
for lr, (theta_exp, loss_exp) in results.items():
    print(f"학습률 {lr:>8}: 최종 MSE = {loss_exp[-1]:.6f}, epochs = {len(loss_exp)}")

### 실험 결과 분석

**관찰 포인트**:

1. **lr = 1e-5 (너무 작음)**:
   - 학습이 매우 느림
   - 50 epoch으로는 수렴하지 않을 수 있음
   - 안정적이지만 비효율적

2. **lr = 0.01 (적절함)**:
   - 안정적으로 빠르게 수렴
   - 조기 종료가 작동
   - 대부분의 경우 좋은 선택

3. **lr = 0.05 (약간 큼)**:
   - 빠르게 수렴하지만 약간 불안정할 수 있음
   - 진동(oscillation) 발생 가능

4. **lr = 1.0 (너무 큼)**:
   - 발산(divergence) 가능성
   - MSE가 증가하거나 NaN 발생
   - 실무에서는 사용 불가

**결론**: 학습률은 모델 학습에서 가장 중요한 하이퍼파라미터 중 하나입니다. 상황에 따라 학습률은 영향이 다르기 때문에 실험을 통해 확인해야합니다.

## 학생용 자가 체크리스트

다음 항목들을 스스로 체크해보세요:

- [ ] 연속형 변수와 범주형 변수를 구분할 수 있다
- [ ] 표준화의 필요성과 방법을 이해했다
- [ ] NumPy의 브로드캐스팅을 활용할 수 있다
- [ ] 정규방정식, lstsq, SVD의 차이를 설명할 수 있다
- [ ] 손실 함수(MSE)가 무엇인지 설명할 수 있다
- [ ] 경사 하강법의 원리를 이해했다
- [ ] 미니배치 학습의 장점을 설명할 수 있다
- [ ] 학습률이 학습에 미치는 영향을 이해했다
- [ ] matplotlib로 학습 과정을 시각화할 수 있다
- [ ] `@`, `.T`, `shape`의 의미를 이해하고 사용할 수 있다

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.